In [ ]:

from eproc_driver import eproc as eproc

from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup

import time
from pathlib import Path
import io
import os
import re
import pandas as pd
import configparser
import pyautogui
import sqlite3
import pyperclip

config = configparser.ConfigParser()
config.read("usuario.txt")
            
# path to config file
LOGIN = config.get("vars", "LOGIN")
SENHA = config.get("vars", "SENHA")
DOWNLOADPATH = config.get("vars", "DOWNLOADPATH")
print("Configurações do usuário importadas.")

# inicializa tudo,  cria um browser Chrome
options=eproc.configura_webdriver(DOWNLOADPATH)
browser=eproc.novo_browser(options)
driver=eproc.novo_webdriver()

Configurações do usuário importadas.


Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


In [ ]:
def trataMinuta(texto, tipo_ato):
    # Regex para capturar o texto a partir do tipo_ato até @NUMEROPROCESSOFORMATADO@
    # O re.escape é usado para escapar caracteres especiais no tipo_ato
    padrao = re.compile(
        rf"{re.escape(tipo_ato)}.*?(@NUMEROPROCESSOFORMATADO@)", 
        re.IGNORECASE | re.DOTALL | re.UNICODE
    )    
    match = padrao.search(texto)    
    if not match:
        return None  # Retorna None se não encontrou o padrão    
    texto_limpo = match.group(0)    
    # Remove tudo depois de @NUMEROPROCESSOFORMATADO@ (inclusive o que vier depois dele)
    texto_limpo = re.sub(r"(@NUMEROPROCESSOFORMATADO@).*", r"\1", texto_limpo, flags=re.DOTALL)    
    # Remove quebras de linha em excesso (mais de 2 quebras viram 2)
    texto_limpo = re.sub(r'\n\s*\n+', '\n\n', texto_limpo)    
    # Remove espaços em excesso nas linhas
    texto_limpo = '\n'.join(linha.strip() for linha in texto_limpo.splitlines())    
    return texto_limpo

def pegaMinuta(driver, cod_minuta):
    pyautogui.click(1000, 600)
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").clear()
    time.sleep(0.2)  # Pequena pausa para segurança
    #insere o código 
    driver.find_element(By.ID, "txtCodigoModelo").click()
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").send_keys(cod_minuta)
    time.sleep(2)
    pyautogui.hotkey('enter')
    time.sleep(4)
    # Localiza o elemento com código
    elemento = driver.find_element(By.PARTIAL_LINK_TEXT, str(cod_minuta))
    # Cria uma cadeia de ações e move o mouse até o elemento
    actions = ActionChains(driver)
    actions.move_to_element(elemento).perform()
    time.sleep(4)
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()
    pyautogui.click(1000, 600)
    pyautogui.hotkey('f5')
    time.sleep(3)  # Pequena pausa para segurança
    #devolve o conteudo
    return conteudo

def gravaTextoMinuta(cod_minuta, texto_minuta):
    conn = sqlite3.connect('minutas.db')
    cursor = conn.cursor()
    query_insert = 'UPDATE minutas SET conteudo = ? WHERE Código = ?;'
    cursor.execute(query_insert, (texto_minuta, cod_minuta))
    conn.commit()
    conn.close()

def pegaProximaMinutaVazia():

    conn = sqlite3.connect('minutas.db')
    cursor = conn.cursor()
    query_select='SELECT "Código", "Tipo de Documento" FROM minutas WHERE conteudo IS NULL LIMIT 1'
    cursor.execute(query_select)
    resultados = cursor.fetchall()
    return resultados

def extrair_tabela_com_selenium(driver: webdriver.Chrome) -> pd.DataFrame:
    """
    Usa Selenium para extrair dados de uma tabela de eventos em uma página
    e os retorna como um DataFrame do Pandas.

    Args:
        driver (webdriver.Chrome): A instância do driver do Selenium já navegando na página.

    Returns:
        pd.DataFrame: Um DataFrame contendo os dados da tabela. Retorna um DataFrame
                      vazio se a tabela não for encontrada.
    """
    try:
        # 1. Espera a tabela estar presente na página antes de interagir com ela (boa prática)
        wait = WebDriverWait(driver, 10) # Espera até 10 segundos
        table_element = wait.until(
            EC.presence_of_element_located((By.ID, 'tblEventos'))
        )
        print("✅ Tabela 'tblEventos' encontrada na página.")

        # 2. Prepara a lista para armazenar os dados e define as colunas
        dados_dos_eventos = []
        colunas = ['Evento', 'Data/Hora', 'Descrição', 'Usuário', 'Documentos']

        # 3. Encontra todas as linhas (tr) dentro do corpo da tabela (tbody)
        rows = table_element.find_element(By.TAG_NAME, 'tbody').find_elements(By.TAG_NAME, 'tr')
        print(f"-> Encontradas {len(rows)} linhas de eventos.")

        # 4. Itera sobre cada linha para extrair os dados das células (td)
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, 'td')
            
            if len(cells) == 6:
                # Extrai o texto de cada célula, limpando espaços em branco
                num_evento = cells[1].text.strip()
                data_hora = cells[2].text.strip()
                
                # O .text do Selenium já lida bem com <br>, mas normalizar os espaços é bom
                descricao = ' '.join(cells[3].text.split())
                
                usuario = cells[4].text.strip()
                
                # Para documentos, encontra todos os links e junta seus textos
                doc_links = cells[5].find_elements(By.CSS_SELECTOR, "a.infraLinkDocumento")
                documentos = ', '.join([link.text.strip() for link in doc_links])

                dados_dos_eventos.append({
                    'Evento': num_evento,
                    'Data/Hora': data_hora,
                    'Descrição': descricao,
                    'Usuário': usuario,
                    'Documentos': documentos
                })

        # 5. Cria o DataFrame do Pandas
        df = pd.DataFrame(dados_dos_eventos, columns=colunas)
        return df

    except TimeoutException:
        print("❌ Erro: A tabela com id 'tblEventos' não foi encontrada na página após 10 segundos.")
        return pd.DataFrame() # Retorna um DataFrame vazio
    except Exception as e:
        print(f"❌ Ocorreu um erro inesperado durante a extração: {e}")
        return pd.DataFrame()



In [ ]:
eproc.entrar_no_processo(driver, 50041251420248210069)


In [70]:

def eventos_para_tabela ():
    return 1

# Localiza a tabela pelo ID e obtém seu inner HTML
tbl_eventos_element = driver.find_element(By.ID, "tblEventos")
html_content = tbl_eventos_element.get_attribute("innerHTML")


# Usa BeautifulSoup para processar o HTML da tabela
soup = BeautifulSoup(html_content, "html.parser")



In [77]:
# Remove a primeira coluna (primeiro <td> de cada <tr>)
for tr in soup.find_all("tr"):
    first_td = tr.find("td")
    if first_td:
        first_td.decompose()

# Na segunda coluna, mantenha só o número e adicione ".0" ao final
for tr in soup.find_all("tr"):
    tds = tr.find_all("td")
    if len(tds) > 0:
        # Extrai apenas o número do texto (antes de qualquer espaço)
        numero = re.match(r"(\d+)", tds[0].get_text(strip=True))
        if numero:
            tds[0].string = f"{numero.group(1)}.0"

# Remove linhas que não tenham número na primeira coluna (Evento)
for tr in soup.find_all("tr"):
    tds = tr.find_all("td")
    if tds:
        evento_text = tds[0].get_text(strip=True) if len(tds) > 0 else ""
        if not re.match(r"^\d+(\.\d+)?$", evento_text):
            tr.decompose()

In [78]:
print(soup.prettify())


<thead>
 <tr>
  <th class="infraTh" title="Evento relevante" width="18">
   <img alt="Evento relevante" src="./imagens/EstrelaAcesa.gif" style="margin-top:1px;margin-left:0px;width: 15px; height: 15px; opacity: 1; border-width: 0pt; "/>
  </th>
  <th class="infraTh" width="56">
   Evento
  </th>
  <th class="infraTh" width="10%">
   Data/Hora
  </th>
  <th class="infraTh">
   Descrição
  </th>
  <th class="infraTh" width="5%">
   Usuário
  </th>
  <th class="infraTh" width="32%">
   Documentos
  </th>
 </tr>
</thead>
<tbody>
 <tr class="infraTrClara infraEventoPrazoAguardando" data-parte="INTERNO" id="trEvento12">
  <td>
   17.0
  </td>
  <td class="infraEventoDescricao">
   <label class="infraEventoDescricao">
    Expedida/certificada a intimação eletrônica
   </label>
   <br/>
   Refer.  ao Evento 11
   <br/>
   (
   <span class="infraEventoPrazoParte" data-parte="AUTOR">
    EXEQUENTE
   </span>
   -  MUNICÍPIO DE SARANDI / RS)
   <br/>
   Prazo: 60 dias Status:AGUARD. ABERTURA
   <

In [ ]:



# Extrai o cabeçalho da tabela (apenas da primeira linha de <tr> que contém <th>)
header = None
for tr in soup.find_all("tr"):
    ths = tr.find_all("th")
    if ths:
        # Pega apenas os textos dos headers, ignora o primeiro (imagem), pega os próximos 5
        header = [th.get_text(strip=True) for th in ths[1:6]]
        break

# Extrai os dados da tabela para uma lista de listas
dados = []
for row in soup.find_all("tr"):
    cells = row.find_all("td")
    if cells:
        # Ignora a primeira célula (imagem), pega as próximas 5
        linha = [cell.get_text(strip=True) for cell in cells[1:6]]
        # Só adiciona se o campo "Evento" (primeiro da linha) for um número
        if linha and linha[0].isdigit():
            dados.append(linha)

# Cria o DataFrame e salva como CSV
df_tabela = pd.DataFrame(dados, columns=header if header else None)

# Converte a coluna "Evento" para float, adicionando ".0" aos valores inteiros
df_tabela["Evento"] = df_tabela["Evento"].astype(float)

# Para cada linha, verifica se há múltiplos links na coluna "Documentos"
novas_linhas = []
for idx, row in df_tabela.iterrows():
    documentos = row["Documentos"]
    # Encontra todos os links <a> na célula correspondente usando BeautifulSoup
    # Primeiro, localiza a linha original no HTML
    tr = soup.find_all("tr")[idx + 1]  # +1 porque header é a primeira linha
    tds = tr.find_all("td")
    if len(tds) >= 6:
        doc_cell = tds[5]
        links = doc_cell.find_all("a")
        if links:
            for i, link in enumerate(links):
                if i > 0:
                    # Cria linha intermediária para cada link extra
                    nova_linha = row.copy()
                    nova_linha["Evento"] = row["Evento"] + (i / 100)  # X.01, X.02, etc.
                    nova_linha["Documentos"] = ""
                    novas_linhas.append(nova_linha)

# Adiciona as linhas intermediárias ao DataFrame
if novas_linhas:
    df_tabela = pd.concat([df_tabela, pd.DataFrame(novas_linhas)], ignore_index=True)
    df_tabela = df_tabela.sort_values(by="Evento").reset_index(drop=True)


In [67]:
df_tabela.to_csv("tabela_eventos_tmp.csv", index=False, encoding="utf-8-sig")
print("Arquivo 'tabela_eventos.csv' criado com sucesso.")

Arquivo 'tabela_eventos.csv' criado com sucesso.


In [79]:
# Salva o conteúdo HTML do BeautifulSoup em um arquivo
with open("tabela_eventos.html", "w", encoding="utf-8") as f:
    f.write(str(soup))
print("Arquivo 'tabela_eventos.html' criado com sucesso.")

Arquivo 'tabela_eventos.html' criado com sucesso.


In [53]:
html_completo = tbl_eventos_element.get_attribute("outerHTML")
print(html_completo)

<table id="tblEventos" class="infraTable table-not-hover results" summary="Eventos" style="width: 99.7%; margin: 0px 2px;">
<thead><tr>
<th width="18" class="infraTh" title="Evento relevante"><img alt="Evento relevante" src="./imagens/EstrelaAcesa.gif" style="margin-top:1px;margin-left:0px;width: 15px; height: 15px; opacity: 1; border-width: 0pt; "></th><th width="56" class="infraTh">Evento</th><th width="10%" class="infraTh">Data/Hora</th><th class="infraTh">Descrição</th><th width="5%" class="infraTh">Usuário</th><th width="32%" class="infraTh">Documentos</th></tr></thead>
<tbody><tr id="trEvento12" class="infraTrClara infraEventoPrazoAguardando" data-parte="INTERNO"><td><a href="javascript:switchRelevanciaEvento('M', '12', 'controlador_ajax.php?acao_ajax=definir_relevancia_evento_processo&amp;numProcesso=50041251420248210069&amp;numSeqEvento=12&amp;tipoGrauEvento=M&amp;hash=565fe26fed21fb23f2e62d980958217c');"><img id="imgMarcarRelevanteM12" name="imgMarcarRelevanteM12" style="margi

In [18]:

# Salva o HTML em um arquivo temporário para o Selenium poder abri-lo
file_path = "temp_table.html"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

try:
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Carrega a página local no navegador
    # os.path.realpath garante que o caminho do arquivo seja absoluto
    full_path = os.path.realpath(file_path)
    driver.get(f"file://{full_path}")
    print(f"Navegador abriu o arquivo local: {full_path}")

    # Chama a função para extrair a tabela
    df_final = extrair_tabela_com_selenium(driver)

    # Imprime o resultado
    if not df_final.empty:
        print("\n--- DataFrame Extraído ---")
        print(df_final)
        
        # Opcional: Salvar em um arquivo CSV
        df_final.to_csv("eventos_extraidos.csv", index=False, encoding='utf-8-sig')
        print("\n✅ DataFrame também foi salvo em 'eventos_extraidos.csv'")

finally:
    # Garante que o navegador e o arquivo temporário sejam fechados/removidos
    if driver:
        driver.quit()
    if os.path.exists(file_path):
        os.remove(file_path)
    print("\nRecursos limpos (navegador fechado e arquivo temporário removido).")

Navegador abriu o arquivo local: C:\wamp64\www\UNICA\temp_table.html
✅ Tabela 'tblEventos' encontrada na página.
-> Encontradas 17 linhas de eventos.

--- DataFrame Extraído ---
  Evento            Data/Hora  \
0     12  17/06/2025 15:46:35   
1     10  02/06/2025 18:49:29   
2      8  15/03/2025 20:48:02   
3      7  21/01/2025 22:56:45   
4      6  20/01/2025 23:55:35   
5      5  11/12/2024 23:59:59   
6      4  01/12/2024 06:58:27   
7      2  29/11/2024 15:32:49   

                                           Descrição  \
0  Expedida/certificada a intimação eletrônica Re...   
1                    Conclusos para decisão/despacho   
2  Juntada de certidão - alteração do prazo - 25/...   
3  Juntada de certidão - alteração do prazo - Mot...   
4  Juntada de certidão - alteração do prazo - Mot...   
5  Confirmada a intimação eletrônica - Refer. ao ...   
6  Expedida/certificada a intimação eletrônica - ...   
7                    Conclusos para decisão/despacho   

                   

## Título



In [ ]:
conn = sqlite3.connect('pet.db')
cursor = conn.cursor()
query_select = 'SELECT COUNT(*) FROM SARANDI WHERE conteudo IS NULL'
cursor.execute(query_select)
conn

In [7]:
#entra no processo 50207613720228210033
processo = 50207613720228210033
eproc.entrar_no_processo(driver,processo)


In [ ]:
#EXECUTA!
conn = sqlite3.connect('minutas.db')
cursor = conn.cursor()
query_select = 'SELECT COUNT(*) FROM minutas WHERE conteudo IS NULL'
cursor.execute(query_select)
resultado = cursor.fetchone()[0]  # pega o número da tupla (ex: (42,) -> 42)
for _ in range(resultado):
    resultados = pegaProximaMinutaVazia()
    for r in resultados:
        try:
            cod_minuta = r[0]
            texto_minuta = pegaMinuta(driver, cod_minuta)
            gravaTextoMinuta(cod_minuta, texto_minuta)  
            print(f"Texto da minuta {cod_minuta} capturado")  
        except:
            print("Erro. Passando para a próxima")

Texto da minuta 10000030467 capturado
Texto da minuta 10000030468 capturado
Texto da minuta 10000030469 capturado
Texto da minuta 10000030470 capturado
Texto da minuta 10000030471 capturado
Texto da minuta 10000030472 capturado
Texto da minuta 10000030473 capturado
Texto da minuta 10000030474 capturado
Texto da minuta 10000030475 capturado
Texto da minuta 10000030476 capturado
Texto da minuta 10000030477 capturado
Texto da minuta 10000030478 capturado
Texto da minuta 10000030479 capturado
Texto da minuta 10000030443 capturado
Texto da minuta 10000030480 capturado
Texto da minuta 10000030481 capturado
Texto da minuta 10000030482 capturado
Texto da minuta 10000030483 capturado
Texto da minuta 10000030484 capturado
Texto da minuta 10000030485 capturado
Texto da minuta 10000030486 capturado
Texto da minuta 10000030487 capturado
Texto da minuta 10000030488 capturado
Texto da minuta 10000030489 capturado
Texto da minuta 10000030490 capturado
Texto da minuta 10000030491 capturado
Texto da min

In [68]:
conn.close()